# SpendKey GenAI Classifier - 01. Data Preparation & Multi-Signal Retrieval

This notebook covers the data preparation and explainable candidate retrieval stages of the pipeline:
1. **Data Ingestion & Integrity Check**: Loads 197 spend transactions and the official 256-node 4-tier taxonomy (`Spendkey_Assignment.xlsx`), validating hierarchy structure.
2. **Explainable Multi-Signal Retrieval**: Demonstrates keyword and boundary-note scoring (L4: +10, Boundary: +8, L3: +7, Token Overlap: +1..+5) with domain synonym expansion (`EPC`, `PLC`, `Simon Jersey`, etc.) without black-box ML vector embeddings.
3. **Batched Prompt Assembly**: Bundles 10 transactions into a grounded prompt with 5 expert procurement few-shot demonstrations and strict JSON schema rules.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
from src.data_processing import load_transactions, load_taxonomy, get_data_quality_report
from prompts.prompts import (
    retrieve_relevant_taxonomy,
    format_taxonomy_for_prompt,
    build_classification_prompt,
    build_batch_classification_prompt,
    SYSTEM_PROMPT
)

EXCEL_PATH = "../data/Spendkey_Assignment.xlsx"
print("Environment initialized successfully.")

Environment initialized successfully.


## 1. Data Ingestion & Quality Validation
Load the original Excel workbook and verify data integrity across transactions and the 4-tier taxonomy hierarchy (`L1 > L2 > L3 > L4`).

In [2]:
transactions_df = load_transactions(EXCEL_PATH)
taxonomy_df = load_taxonomy(EXCEL_PATH)
print(f"Loaded {len(transactions_df)} transactions and {len(taxonomy_df)} taxonomy categories.")

quality = get_data_quality_report(transactions_df, taxonomy_df)
print("\nData Quality Report:")
for k, v in quality.items():
    print(f"  - {k}: {v}")

Loaded 197 transactions and 256 taxonomy categories.

Data Quality Report:
  - n_transactions: 197
  - n_taxonomy_leaf_nodes: 256
  - n_unique_l1_segments: 11
  - missing_descriptions: 0
  - duplicate_transaction_rows: 1
  - n_boundary_notes: 32


C:\Users\prath\AppData\Roaming\Python\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
print("Sample Spend Transactions (First 5):")
display(transactions_df.head(5)[["transaction_id", "spend_description", "vendor", "source_type"]])

print("\nTaxonomy Distribution by L1 Segment:")
display(taxonomy_df["L1"].value_counts().to_frame(name="Total Categories"))

Sample Spend Transactions (First 5):


,transaction_id,spend_description,vendor,source_type
0,1,"Microsoft 365 E3 – annual renewal, 800 seats",Microsoft,Vendor invoice / PO
1,2,"Microsoft 365 E3 – annual renewal, 800 seats",Microsoft,Vendor invoice / PO
2,3,"Dell Latitude 5540 – 120-unit refresh, 3yr war...",Dell Technologies,Vendor invoice / PO
3,4,"Okta Identity Cloud – 2,000-user SSO, annual SaaS",Okta,Vendor invoice / PO
4,5,Zoom Enterprise – 500-seat annual licence,Zoom,Vendor invoice / PO



Taxonomy Distribution by L1 Segment:


,Total Categories
L1,
IT & Technology,39
Professional Services,30
Facilities & Property,29
Marketing & Communications,29
HR & Workforce,28
Finance & Insurance,20
Logistics & Supply Chain,20
MRO & Engineering,19
Utilities & Energy,18


## 2. Explainable Multi-Signal Taxonomy Retrieval
The system uses an explainable multi-signal keyword and boundary scoring formula:
- **Exact L4 Match**: +10 pts
- **Boundary Note Match**: +8 pts
- **Exact L3 Match**: +7 pts
- **Token Overlap Weights**: L4 (+5), L3 (+3), L2 (+2), L1 (+1)
- **Domain Synonyms & Vendor Hints**: Expands procurement acronyms (e.g. `EPC`, `PLC`, `Simon Jersey`, `Deel`).

Let's test candidate retrieval on representative real-world edge cases.

In [4]:
test_cases = [
    {"spend_description": "Power BI Premium monthly SaaS subscription", "vendor": "Microsoft"},
    {"spend_description": "Energy Performance Certificates (EPC) commercial surveys", "vendor": "Bureau Veritas"},
    {"spend_description": "Siemens S7-1500 PLC controller upgrade module", "vendor": "Siemens Industry"},
    {"spend_description": "Staff hospitality uniforms and protective tabards", "vendor": "Simon Jersey"}
]

print("MULTI-SIGNAL CANDIDATE RETRIEVAL RESULTS:")
print("=" * 75)
for tc in test_cases:
    query = f"{tc['spend_description']} {tc['vendor']}"
    candidates_df = retrieve_relevant_taxonomy(query, taxonomy_df, top_n=5)
    top = candidates_df.iloc[0]
    print(f"Query:  {tc['spend_description']} (Vendor: {tc['vendor']})")
    print(f"Top L4: {top['L4']}")
    print(f"Path:   {top['full_path']}")
    if top.get('boundary_note') and str(top['boundary_note']).strip():
        print(f"Note:   {top['boundary_note']}")
    print("-" * 75)

MULTI-SIGNAL CANDIDATE RETRIEVAL RESULTS:
Query:  Power BI Premium monthly SaaS subscription (Vendor: Microsoft)
Top L4: Data Warehousing Platforms
Path:   IT & Technology > Software > Data & Analytics > Data Warehousing Platforms
Note:   Cloud data warehouse (Snowflake, BigQuery). Not generic SaaS.
---------------------------------------------------------------------------
Query:  Energy Performance Certificates (EPC) commercial surveys (Vendor: Bureau Veritas)
Top L4: Energy Performance Certificates
Path:   Facilities & Property > Compliance & Environment > Statutory Compliance > Energy Performance Certificates
---------------------------------------------------------------------------
Query:  Siemens S7-1500 PLC controller upgrade module (Vendor: Siemens Industry)
Top L4: Electrical Components & Fuses
Path:   MRO & Engineering > Spare Parts & Components > Electrical Parts > Electrical Components & Fuses
---------------------------------------------------------------------------
Quer

## 3. Batched Prompt Construction
Transactions are grouped into batches of 10 to reduce API calls by 90% (~197 calls down to ~20 requests).
Each batch includes candidate categories with official boundary notes, few-shot examples, and JSON schema constraints.

In [5]:
batch_sample = transactions_df.head(10).to_dict("records")
contexts = []
for row in batch_sample:
    query = f"{row['spend_description']} {row['vendor']}"
    sub = retrieve_relevant_taxonomy(query, taxonomy_df, top_n=7)
    paths = []
    for _, tax_row in sub.iterrows():
        p = tax_row['full_path']
        note = tax_row['boundary_note']
        if note and str(note).strip():
            paths.append(f"{p}  [Boundary note: {str(note).strip()}]")
        else:
            paths.append(p)
    contexts.append({"transaction_id": int(row["transaction_id"]), "paths": paths})

prompt_text = build_batch_classification_prompt(batch_sample, contexts)
print(f"Generated Batched Prompt for 10 Transactions ({len(prompt_text)} chars).")
print("\n--- PROMPT PREVIEW (First 750 characters) ---\n")
print(prompt_text[:750] + "\n...\n[Candidate lists & few-shot examples omitted for preview]")

Generated Batched Prompt for 10 Transactions (14234 chars).

--- PROMPT PREVIEW (First 750 characters) ---

You are a senior procurement spend classification analyst classifying transactions into the SpendKey taxonomy.

KEY PROCUREMENT CLASSIFICATION RULES:
1. Use ONLY the candidate taxonomy paths supplied under each transaction.
2. Select the most specific valid L4 commodity.
3. Classify based primarily on what is being purchased, using vendor name as context.
4. Apply [Boundary notes] as binding authoritative rules (e.g. IT contractors go to HR & Workforce).
5. Bundled contracts (e.g. IFM bundled cleaning/catering, PFI Unitary Charge) or unmapped medical locum staffing must be assigned the closest category with human_review_required=true.
6. Set confidence_level to HIGH, MEDIUM, or LOW based on evidence.
7. Return exactly one classification for
...
[Candidate lists & few-shot examples omitted for preview]
